In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_data=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_data=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
print(train_data.shape,test_data.shape)
print(train_data.head())
print(train_data.info())

In [ ]:
X = train_data.drop(['Id','SalePrice'],axis=1)
y = train_data['SalePrice']
y=np.log1p(y)
print(X.shape)
print(y.shape)

In [ ]:
all=pd.concat([X,test_data],axis=0,ignore_index=True)
print(all.shape)
print(all.head())

In [ ]:
all=all.dropna(axis=1,thresh=2000)
print(all.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
numerical_features = all.select_dtypes(exclude=['object']).columns.tolist()
categorical_features = all.select_dtypes(include=['object']).columns.tolist()

In [ ]:
all[categorical_features] = all[categorical_features].fillna('ydq')
all[numerical_features] = all[numerical_features].fillna(all[numerical_features].mean())

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        
        ("num", StandardScaler(), numerical_features),
        
        
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features) 
    ]
)

In [ ]:
X=all.iloc[:1460,:]
test=all.iloc[1460:,:]
print(X.shape,test.shape)
X=preprocessor.fit_transform(X)
test=preprocessor.transform(test)

In [ ]:

#from sklearn.linear_model import LinearRegression
#model=LinearRegression()
#model.fit(X,y)
from sklearn.neighbors import KNeighborsRegressor
model=KNeighborsRegressor(n_neighbors=10,weights='distance')
model.fit(X,y)



In [ ]:
'''
from sklearn.model_selection import train_test_split,GridSearchCV
model=KNeighborsRegressor()
param_grid={'n_neighbors':list(range(1,40)),'weights':['distance']}
grid_search_cv=GridSearchCV(estimator=model,param_grid=param_grid,cv=10)
grid_search_cv.fit(X,y)
res=pd.DataFrame(grid_search_cv.cv_results_).to_string()
print(res)
'''

In [ ]:
result=model.predict(test)
result=np.expm1(result)

In [ ]:
index=test_data['Id']
submission=pd.DataFrame({'Id':index,'SalePrice':result})
submission.to_csv('submission.csv', index=False)